In [11]:
import pandas as pd
import numpy as np

### This is a notebook used to convert the cleaned PIR data for House 28 in LEEDR into an occupancy schedule and heating pattern for use in 10min timesteps 

In [12]:
h28 = pd.read_csv("/workspaces/CUBES/exp/jack/paper/thermostat_experiment/SI/test_case_H28/H28_cleaned_2013.csv")

In [15]:

h28['UTC_Time'] = pd.to_datetime(h28['UTC_Time'])

mapping = {'H28_Backroom(Down stairs)': 'backroom',
           'H28_Bathroom 1(Upstairs)': 'bathroom',
           'H28_Hall(Down stairs)':'hall_downstairs',
           'H28_Front Room(Down stairs)':'front_room',
           'H28_Kitchen(Down stairs)':'kitchen',
           'H28_Bedroom 3(Upstairs)':'bedroom_3',
           'H28_Bedroom 1(Upstairs)': 'bedroom_1',
           'H28_Bedroom 2(Upstairs)': 'bedroom_2',
           }

h28 = h28.rename(columns=mapping)


h28["hall_upstairs"] = h28["hall_downstairs"]



In [16]:
h28

,UTC_Time,kitchen,bathroom,bedroom_3,hall_downstairs,front_room,backroom,bedroom_2,bedroom_1,hall_upstairs
0,2013-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2013-01-01 00:01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2013-01-01 00:02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2013-01-01 00:03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2013-01-01 00:04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525596,2013-12-31 23:56:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525597,2013-12-31 23:57:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525598,2013-12-31 23:58:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Generate heating pattern from occupancy data

In [17]:

# Assuming df is your original DataFrame
df_copy = h28.copy()

# Convert 'UTC_Time' to datetime to work with times
df_copy['UTC_Time'] = pd.to_datetime(df_copy['UTC_Time'])

# Create a boolean mask for the time between 23:00 and 07:00
time_mask = (df_copy['UTC_Time'].dt.hour >= 23) | (df_copy['UTC_Time'].dt.hour < 7)

# Define a function to apply the heating rule
def apply_heating_rule(series, time_mask):
    heating = np.zeros_like(series)  # Initial heating schedule, all off (0)
    for i in range(5, len(series)):  # Start from 5th element due to 5-minute window
        if time_mask[i]:  # If time is between 23:00 and 07:00, heating is off
            heating[i] = 0
        elif series[i] > 0:  # If current value is greater than 0, heating is on for 5 mins
            heating[i-4:i+1] = 1
        elif np.all(series[i-4:i] == 0):  # If no activity in the last 5 mins, heating off
            heating[i] = 0
    return heating

# Apply the heating rule to all rooms
rooms = ['backroom', 'bathroom', 'front_room', 'hall_downstairs', 'bedroom_2', 'kitchen', 'bedroom_1', 'bedroom_3', 'hall_upstairs']

for room in rooms:
    df_copy[room] = apply_heating_rule(h28[room].values, time_mask)

# Now df_copy contains your desired heating schedule


In [18]:
df_copy

,UTC_Time,kitchen,bathroom,bedroom_3,hall_downstairs,front_room,backroom,bedroom_2,bedroom_1,hall_upstairs
0,2013-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2013-01-01 00:01:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2013-01-01 00:02:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2013-01-01 00:03:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2013-01-01 00:04:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
525595,2013-12-31 23:55:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525596,2013-12-31 23:56:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525597,2013-12-31 23:57:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525598,2013-12-31 23:58:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
# Resample the dataframe in 10-minute intervals, applying the max function to each group
df_resampled = df_copy.resample('10T', on='UTC_Time').max()

# If you need to reset the index to make 'UTC_Time' a normal column again
df_resampled = df_resampled.reset_index()

df_resampled = df_resampled.set_index("UTC_Time")
df_resampled = df_resampled.clip(upper=1)
df_resampled.to_csv("heating_pattern_h28.sch")

/tmp/ipykernel_19035/299016176.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = df_copy.resample('10T', on='UTC_Time').max()


In [20]:
df_resampled

,kitchen,bathroom,bedroom_3,hall_downstairs,front_room,backroom,bedroom_2,bedroom_1,hall_upstairs
UTC_Time,,,,,,,,,
2013-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-01-01 00:40:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
2013-12-31 23:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-12-31 23:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2013-12-31 23:30:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Generate 10-minute occupancy pattern

In [21]:
# Resample the dataframe in 10-minute intervals, applying the max function to each group
df_resampled = h28.resample('10T', on='UTC_Time').max()

# If you need to reset the index to make 'UTC_Time' a normal column again
df_resampled = df_resampled.reset_index()

df_resampled = df_resampled.set_index("UTC_Time")
df_resampled = df_resampled.clip(upper=1)
df_resampled.to_csv("occupancy_pattern_h28.sch")

/tmp/ipykernel_19035/4289872751.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df_resampled = h28.resample('10T', on='UTC_Time').max()
